# colab_07 — Stratified subsampling to 100k per dataset

## Motivation

Session 15 showed DPT fails on the integrated Harmony object. Root causes: (1) ~100× cell imbalance between Bhaduri 2020 organoids (~242k) and Zhong 2018 fetal (~2.4k); (2) disconnected graph components; (3) stale `iroot` propagation through `.copy()`.

Session 16 pivoted: replace Zhong with **Bhaduri 2021** (atlas of cortical arealization; ~396k fetal cells). Same lab, same 10x v2 chemistry as Bhaduri 2020 organoids.

Session 17 downloaded Bhaduri 2021 and saved `bhaduri_2021_raw.h5ad` (396,186 × 33,694) on Drive.

**This notebook builds a balanced 1:1 input for integration:** stratified subsample both datasets to 100k cells each. Stratum floor = 200 cells (hard-drop, no upsampling — upsampling duplicates cells at zero distance in the kNN graph, which is exactly the pathology that broke DPT). Per-stratum targets = proportional to cleaned stratum size (preserves natural composition).

Stratification axes:
- **Bhaduri 2020** (organoids): `protocol × age_week` (organoids have no anatomical area).
- **Bhaduri 2021** (fetal): `age_gw × cell_type_coarse`. `area_ucsc` deliberately left out — it would only apply to the fetal side (asymmetric), fragment strata from ~120 → ~1,200 (most below the 200 floor), and areal identity is a Phase-2 question.

## Section 0 — Setup

Install `scanpy` (fresh Colab kernel) and mount Drive for persistent h5ad storage.

In [ ]:
!pip install -q scanpy
from google.colab import drive
drive.mount('/content/drive')

## Section 1 — Load Bhaduri 2020 and inspect metadata

Starting file is `bhaduri_2020_clustered.h5ad` — contains raw counts in `adata.raw`, Leiden labels, and sample prefixes on barcodes. Integration-stage transforms (HVG selection, PCA, Harmony) will be redone downstream on the balanced input, so we start from the clustered h5ad rather than the dense-scaled preprocessed one (which was ~32 GB anyway and has been deleted).

Key question for subsampling: **does the `obs` already contain `protocol` and `age` columns?** If not, we must derive them from the `sample` prefix.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad')
print("Shape:", adata.shape)
print("\nobs columns:", list(adata.obs.columns))
print("\nFirst 5 obs rows:")
print(adata.obs.head())
print("\nUnique samples (first 10):", adata.obs['sample'].unique()[:10] if 'sample' in adata.obs.columns else "NO 'sample' COLUMN")
print("\nSample count:", adata.obs['sample'].nunique() if 'sample' in adata.obs.columns else "N/A")

### Finding

- Shape: **241,776 × 16,774**.
- `obs` columns: `n_genes_by_counts`, `total_counts`, `total_counts_mt`, `pct_counts_mt`, `n_genes`, `sample`, `leiden`. **No `protocol`, no `age`** — must be derived.
- 37 unique samples. Names like `H1SWeek3`, `H28126SWeek5`, `YH10PWeek10` — clearly encode **`{cell_line}{protocol_letter}Week{N}`** where protocol ∈ {S, X, P} (matches the paper's 3-protocol design).

## Section 2 — First parsing attempt

Hypothesis: every sample name follows the clean pattern `^{line}{[SXP]}Week{\d+}$`. One regex should cover all 37.

In [ ]:
import re
pattern = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])Week(?P<age>\d+)$')

all_samples = adata.obs['sample'].unique().tolist()
parsed, failed = [], []
for s in all_samples:
    m = pattern.match(s)
    if m:
        parsed.append((s, m.group('line'), m.group('protocol'), int(m.group('age'))))
    else:
        failed.append(s)

print(f"Parsed: {len(parsed)}/{len(all_samples)}")
print(f"Failed: {failed if failed else 'none'}")

import pandas as pd
df = pd.DataFrame(parsed, columns=['sample', 'line', 'protocol', 'age_week'])
print("\nProtocol counts (samples):", df['protocol'].value_counts().to_dict())
print("Age counts (samples):",       df['age_week'].value_counts().sort_index().to_dict())
print("Line counts (samples):",      df['line'].value_counts().to_dict())
print("\nProtocol × age grid (samples):")
print(df.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))

### Finding

**28/37 parse cleanly. 9 fail.** Two pattern families escape the regex:

1. **`H28126S2Week{5,8,10}`** — cell line `H28126` with `S2` designation (likely protocol S, variant/batch 2). The trailing digit breaks the `{protocol_letter}Week` pattern.
2. **`Week{5,8,10}{S,P}`** — reverse-order naming with no cell-line prefix.

Nine samples sounds small, but we don't know yet how many *cells* they represent — need to check before deciding how careful to be.

## Section 3 — Investigate failed samples

If the 9 unmatched samples carry only a handful of cells, we could drop them. If they carry tens of thousands, we must parse them properly.

In [ ]:
failed_samples = ['H28126S2Week8', 'H28126S2Week5', 'Week5S', 'Week8S',
                  'Week8P', 'Week5P', 'H28126S2Week10', 'Week10S', 'Week10P']
print("Cells per unmatched sample:")
print(adata.obs['sample'].value_counts().loc[failed_samples])
print(f"\nTotal unmatched cells: {adata.obs['sample'].isin(failed_samples).sum()}")
print(f"Of grand total: {adata.obs['sample'].isin(failed_samples).sum() / len(adata) * 100:.1f}%")

### Finding

**57,718 cells (23.9% of the dataset)** are in the 9 unmatched samples. Too large to drop. We must extend the parser.

### Decisions on the two edge-case families

**`H28126S2Week*` (3 samples, 16,053 cells) — fold `S2` into `S`.**
- Our stratification axis is *protocol* for balancing against fetal maturation, not batch-effect analysis.
- Folding matches the paper's stated 3-protocol design.
- Alternative (keep S2 separate) would fragment strata and add a 4th protocol level that isn't biologically distinct.

**`Week{N}{S,P}` (6 samples, 41,665 cells) — label cell line as `unknown`; parse protocol + age normally.**
- Cell line is not a stratification axis, so its absence is acceptable.
- Protocol + age are what we actually need.

## Section 4 — Final parser

Three patterns, tried in order of specificity:

1. `pat_variant` — `{line}{[SXP]}\d+Week{N}` — catches `H28126S2Week*`.
2. `pat_standard` — `{line}{[SXP]}Week{N}` — original clean pattern.
3. `pat_reverse` — `Week{N}{[SXP]}` — catches `Week{N}{S,P}`; line labeled `unknown`.

We then map `(protocol, age_week)` onto every cell and compute the **cell-level grid**, which is what matters for stratification sizing.

In [ ]:
import re

pat_standard = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])Week(?P<age>\d+)$')
pat_variant  = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])\d+Week(?P<age>\d+)$')
pat_reverse  = re.compile(r'^Week(?P<age>\d+)(?P<protocol>[SXP])$')

def parse_sample(s):
    for pat in (pat_variant, pat_standard):  # try variant first (more specific)
        m = pat.match(s)
        if m:
            return m.group('line'), m.group('protocol'), int(m.group('age'))
    m = pat_reverse.match(s)
    if m:
        return 'unknown', m.group('protocol'), int(m.group('age'))
    return None

all_samples = adata.obs['sample'].unique().tolist()
parsed, failed = [], []
for s in all_samples:
    r = parse_sample(s)
    if r:
        parsed.append((s, *r))
    else:
        failed.append(s)

import pandas as pd
df = pd.DataFrame(parsed, columns=['sample', 'line', 'protocol', 'age_week'])

print(f"Parsed: {len(parsed)}/{len(all_samples)}")
print(f"Failed: {failed if failed else 'none'}")
print("\nProtocol × age grid (SAMPLES):")
print(df.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))

# Build two separate mappings (tuple-map on a Categorical series triggers a pandas MultiIndex error)
protocol_map = dict(zip(df['sample'], df['protocol']))
age_map      = dict(zip(df['sample'], df['age_week']))

sample_str = adata.obs['sample'].astype(str)
obs_tmp = pd.DataFrame({
    'protocol': sample_str.map(protocol_map),
    'age_week': sample_str.map(age_map),
})

print("\nProtocol × age grid (CELLS):")
print(obs_tmp.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))
print(f"\nTotal cells with (protocol, age): {obs_tmp.dropna().shape[0]} / {len(obs_tmp)}")

### Finding

**37/37 samples parse. 241,776 / 241,776 cells mapped.**

**Cell-level grid:**

| protocol \ age_week | 3 | 5 | 8 | 10 | 15 | 24 |
|---|---|---|---|---|---|---|
| P | 0 | 14,228 | 16,697 | 13,367 | 0 | 1,591 |
| S | 18,017 | 45,757 | 29,567 | 48,183 | 2,718 | 2,914 |
| X | 20,321 | 10,126 | 2,427 | 15,863 | 0 | 0 |

**Stratum health:** 14 populated strata out of 18 possible (4 empty: P×3, P×15, X×15, X×24). Lowest populated stratum = X×8 at **2,427 cells** — an order of magnitude above the 200-cell floor. **The hard-drop rule removes 0 cells from Bhaduri 2020.**

**Rough per-protocol share at 100k target (proportional):** S ≈ 63k, P ≈ 19k, X ≈ 19k. Reflects the true cohort composition — S was run in more cell lines across more timepoints.

## Next

- Section 5 (TODO): stratified subsample Bhaduri 2020 → 100k, save `bhaduri_2020_100k.h5ad`.
- Section 6 (TODO): load Bhaduri 2021 raw, filter `cell_type_coarse == "Outlier"` and `cell_type == "0"` (~80k cells), then stratified subsample → 100k, save `bhaduri_2021_100k.h5ad`.
- Section 7 (TODO): joint sanity checks (shape parity, gene-space overlap preview for integration).